
# HuberRidgeAIME — Main Reproducibility Notebook (Corrected Inverse-Map Implementation)

This notebook supersedes the earlier main-analysis implementation for the current manuscript revision.  Every AIME-family estimator consistently solves

\[
X \approx Y A^\top,
\qquad
A^\top=(Y^\top W Y+\lambda I_C)^{-1}Y^\top W X.
\]

It regenerates the **main/core reproducibility outputs** that are not specific to the separate Reviewer 5 notebook:

1. dataset validity and missingness audit;
2. clean AIME-family benchmark under the corrected inverse orientation;
3. controlled real-data stress benchmark for all four AIME-family variants;
4. paired condition-cluster bootstrap comparisons;
5. Australian Credit complete-case versus imputation comparison;
6. equation-consistent runtime scaling;
7. figures, LaTeX tables, provenance, validation report, and ZIP package.

The separate notebook `HuberRidgeAIME_Reviewer5_FullyCorrected_Experiments_FINAL.ipynb` remains the source for the known-ground-truth faithfulness, detailed HRA-versus-RidgeAIME, hyperparameter-sensitivity, LIME/TreeSHAP, and spectral-filter experiments requested by Reviewer 5.

> **Important:** Run this notebook from the first cell in a fresh runtime.  The default configuration is the full run.  Use quick-test mode only as a smoke test, never for manuscript numbers.


In [1]:

# USER CONFIGURATION — RUN THIS CELL FIRST
USER_OUTPUT_DIR = "./output/main_repro_corrected"
USER_QUICK_TEST = False
USER_FORCE_RECOMPUTE = True
USER_AUTO_INSTALL = True

# Core experiments retained for the final manuscript package.
USER_RUN_CLEAN_BENCHMARK = True
USER_RUN_FULL_STRESS = True
USER_RUN_MISSING_DATA = True
USER_RUN_RUNTIME = True


In [2]:

# %% [configuration and reproducibility]
import os, sys, io, json, math, time, zipfile, platform, warnings, urllib.request, hashlib, shutil, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.special import softmax
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, log_loss, average_precision_score
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

PIPELINE_VERSION = "main_corrected_inverse_map_2026-08-31_v2"
MAIN_SEED = 42
np.random.seed(MAIN_SEED)

QUICK_TEST = bool(globals().get("USER_QUICK_TEST", False))
FORCE_RECOMPUTE = bool(globals().get("USER_FORCE_RECOMPUTE", True))
AUTO_INSTALL = bool(globals().get("USER_AUTO_INSTALL", True))
RUN_CLEAN_BENCHMARK = bool(globals().get("USER_RUN_CLEAN_BENCHMARK", True))
RUN_FULL_STRESS = bool(globals().get("USER_RUN_FULL_STRESS", True))
RUN_MISSING_DATA = bool(globals().get("USER_RUN_MISSING_DATA", True))
RUN_RUNTIME = bool(globals().get("USER_RUN_RUNTIME", True))

PROJECT_ROOT = Path.cwd()
_output = Path(globals().get("USER_OUTPUT_DIR", "./output/main_repro_corrected"))
if QUICK_TEST and str(_output).rstrip("/").endswith("main_repro_corrected"):
    _output = Path(str(_output).rstrip("/") + "_quick")
MAIN_OUTDIR = _output.resolve()
MAIN_DATADIR = MAIN_OUTDIR / "data"
MAIN_TABLEDIR = MAIN_OUTDIR / "tables"
MAIN_FIGDIR = MAIN_OUTDIR / "figures"
MAIN_LOGDIR = MAIN_OUTDIR / "logs"
for p in [MAIN_OUTDIR, MAIN_DATADIR, MAIN_TABLEDIR, MAIN_FIGDIR, MAIN_LOGDIR]:
    p.mkdir(parents=True, exist_ok=True)

DATASETS = ["breast_cancer", "credit_approval", "har"]
LEARNERS = ["lgbm", "svm", "mlp"]
MAX_N_PER_DATASET = {"breast_cancer": None, "credit_approval": None, "har": 3000}
MAX_CLONES = 100
AIME_METHODS = ["AIME", "HuberAIME", "RidgeAIME", "HuberRidgeAIME"]

DEFAULT_RIDGE_LAMBDA = 1e-2
DEFAULT_HUBER_DELTA = 1.0
RESIDUAL_SCALE_MODE = "rms"
TOP_K = 10
IRRELEVANT_DECOY_COUNT = 3

CLEAN_REPEATS = 3
CLEAN_BOOTSTRAPS = 8
CLEAN_DECOY_TRIALS = 3

STRESS_REPEATS = 3
STRESS_OUTLIER_LEVELS = [0.00, 0.05, 0.10]
STRESS_CLONE_FRACS = [0.00, 0.25, 0.50]
STRESS_BOOTSTRAPS = 5
STRESS_DECOY_TRIALS = 3
CLUSTER_BOOTSTRAP_RESAMPLES = 10000

MISSING_REPEATS = 3
MISSING_BOOTSTRAPS = 5
MISSING_DECOY_TRIALS = 3

RUNTIME_N_VALUES = [500, 1000, 2000, 4000]
RUNTIME_D_VALUES = [20, 100, 300]
RUNTIME_REPEATS = 3

if QUICK_TEST:
    DATASETS = ["breast_cancer"]
    LEARNERS = ["lgbm"]
    MAX_N_PER_DATASET = {"breast_cancer": 300}
    MAX_CLONES = 10
    CLEAN_REPEATS = 1
    CLEAN_BOOTSTRAPS = 2
    CLEAN_DECOY_TRIALS = 2
    STRESS_REPEATS = 1
    STRESS_OUTLIER_LEVELS = [0.10]
    STRESS_CLONE_FRACS = [0.50]
    STRESS_BOOTSTRAPS = 2
    STRESS_DECOY_TRIALS = 2
    MISSING_REPEATS = 1
    RUN_MISSING_DATA = False  # smoke test avoids network-dependent credit data
    MISSING_BOOTSTRAPS = 2
    MISSING_DECOY_TRIALS = 2
    RUNTIME_N_VALUES = [200, 500]
    RUNTIME_D_VALUES = [20, 80]
    RUNTIME_REPEATS = 1
    # Offline smoke tests skip the network-dependent Australian Credit comparison.
    RUN_MISSING_DATA = False

print("Pipeline version:", PIPELINE_VERSION)
print("Output directory:", MAIN_OUTDIR)
print("Quick test:", QUICK_TEST)
print("Datasets:", DATASETS)
print("Learners:", LEARNERS)
print("Inverse orientation: X ≈ Y A^T")
print("Default lambda:", DEFAULT_RIDGE_LAMBDA)
print("Default delta:", DEFAULT_HUBER_DELTA)
print("Huber residual: row RMS input-space residual")


Pipeline version: main_corrected_inverse_map_2026-08-31_v2
Output directory: /Users/takafumi/Documents/Python/HuberRidgeAIME/notebooks/output/main_repro_corrected
Quick test: False
Datasets: ['breast_cancer', 'credit_approval', 'har']
Learners: ['lgbm', 'svm', 'mlp']
Inverse orientation: X ≈ Y A^T
Default lambda: 0.01
Default delta: 1.0
Huber residual: row RMS input-space residual


In [3]:

# %% [dependency check]
def ensure_lightgbm():
    try:
        from lightgbm import LGBMClassifier
        return LGBMClassifier, True, ""
    except Exception as first_error:
        if AUTO_INSTALL:
            print("Installing LightGBM...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lightgbm"])
            try:
                from lightgbm import LGBMClassifier
                return LGBMClassifier, True, ""
            except Exception as second_error:
                return None, False, f"{first_error}; after install: {second_error}"
        return None, False, str(first_error)

LGBMClassifier, LGBM_AVAILABLE, LGBM_ERROR = ensure_lightgbm()
if not LGBM_AVAILABLE and not QUICK_TEST:
    raise RuntimeError(f"LightGBM is required for the full run: {LGBM_ERROR}")
print("LightGBM available:", LGBM_AVAILABLE)


LightGBM available: True


## Shared helpers, data loading, stress generation, learners, and equation-consistent inverse-map estimators

In [4]:

# %% [file/table helpers and versioned cache]

def flatten_columns(df):
    """Return a copy with stable one-line column names for CSV/LaTeX export."""
    out = df.copy()
    if isinstance(out.columns, pd.MultiIndex):
        names = []
        for tup in out.columns.to_flat_index():
            parts = [str(x) for x in tup if str(x) not in {"", "None"}]
            names.append("__".join(parts))
        out.columns = names
    else:
        out.columns = [str(c) for c in out.columns]
    return out

def save_csv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    out = df.copy()
    if "pipeline_version" not in out.columns:
        out.insert(0, "pipeline_version", PIPELINE_VERSION)
    out.to_csv(path, index=False)
    print("[csv]", path)
    return path

def load_versioned_csv(path):
    path = Path(path)
    if not path.exists() or FORCE_RECOMPUTE:
        return None
    df = pd.read_csv(path)
    if "pipeline_version" not in df.columns or not (df["pipeline_version"] == PIPELINE_VERSION).all():
        print("[cache ignored: version mismatch]", path)
        return None
    print("[cache]", path)
    return df

def save_table_bundle(df, stem, caption, label, index=False):
    csv_path = MAIN_DATADIR / f"{stem}.csv"
    tex_path = MAIN_TABLEDIR / f"{stem}.tex"
    flat = flatten_columns(df)
    save_csv(flat, csv_path)
    tex_df = flat.drop(columns=["pipeline_version"], errors="ignore")
    tex = tex_df.to_latex(
        index=index, escape=False, caption=caption, label=label, position="t"
    )
    tex_path.write_text(tex, encoding="utf-8")
    print("[tex]", tex_path)
    return csv_path, tex_path

def file_sha256(path, block_size=2**20):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            block = f.read(block_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()

def make_unique(names):
    seen, out = {}, []
    for name in map(str, names):
        if name not in seen:
            seen[name] = 0
            out.append(name)
        else:
            seen[name] += 1
            out.append(f"{name}__dup{seen[name]}")
    return out

def cosine_safe(a, b):
    a = np.asarray(a, float).ravel()
    b = np.asarray(b, float).ravel()
    mask = np.isfinite(a) & np.isfinite(b)
    a, b = a[mask], b[mask]
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return np.nan if denom < 1e-12 else float(np.dot(a, b) / denom)

def spearman_safe(a, b):
    a = np.asarray(a, float).ravel()
    b = np.asarray(b, float).ravel()
    mask = np.isfinite(a) & np.isfinite(b)
    a, b = a[mask], b[mask]
    if len(a) < 2 or np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    return float(stats.spearmanr(a, b).correlation)

def topk_set(scores, k=10):
    scores = np.asarray(scores, float)
    if scores.size == 0:
        return set()
    k = min(int(k), scores.size)
    return set(np.argpartition(-scores, k - 1)[:k].tolist())

def topk_jaccard(a, b, k=10):
    A, B = topk_set(a, k), topk_set(b, k)
    return np.nan if not (A | B) else len(A & B) / len(A | B)

def global_strength(A):
    return np.linalg.norm(np.asarray(A, float), axis=1)

def cluster_bootstrap_ci(cluster_values, n_resamples=10000, confidence=0.95, seed=42):
    values = np.asarray(cluster_values, float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return dict(mean=np.nan, median=np.nan, ci_low=np.nan, ci_high=np.nan, n_clusters=0)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(values), size=(n_resamples, len(values)))
    boot = values[idx].mean(axis=1)
    alpha = (1 - confidence) / 2
    return dict(
        mean=float(values.mean()),
        median=float(np.median(values)),
        ci_low=float(np.quantile(boot, alpha)),
        ci_high=float(np.quantile(boot, 1 - alpha)),
        n_clusters=int(len(values)),
    )

def wilcoxon_two_sided(values):
    values = np.asarray(values, float)
    values = values[np.isfinite(values)]
    if len(values) < 2 or np.allclose(values, 0):
        return np.nan, 1.0
    stat, p = stats.wilcoxon(values, alternative="two-sided", zero_method="wilcox")
    return float(stat), float(p)


In [5]:

# %% [data loading]
def _download_bytes(urls, timeout=180, retries=3, sleep=2):
    if isinstance(urls, str):
        urls = [urls]
    last = None
    for url in urls:
        for _ in range(retries):
            try:
                with urllib.request.urlopen(url, timeout=timeout) as r:
                    return r.read()
            except Exception as e:
                last = e
                time.sleep(sleep)
    raise last if last is not None else RuntimeError("download failed")

def load_breast_cancer_dataset():
    data = load_breast_cancer()
    X = pd.DataFrame(data.data, columns=make_unique(data.feature_names))
    y = pd.Series(data.target.astype(int), name="target")
    meta = {
        "dataset": "breast_cancer",
        "raw_missing_cells": int(X.isna().sum().sum()),
        "source": "sklearn/UCI WDBC",
    }
    return X, y, meta

def load_credit_raw():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/credit-screening/crx.data"
    raw = pd.read_csv(url, header=None, na_values="?")
    raw.columns = [f"A{i+1}" for i in range(raw.shape[1] - 1)] + ["target"]
    return raw

def process_credit(protocol="impute"):
    raw = load_credit_raw()
    proc = raw.dropna().copy() if protocol == "complete_case" else raw.copy()
    y = proc["target"].map({"+": 1, "-": 0}).astype(int)
    X = proc.drop(columns=["target"]).copy()
    cat_cols = [c for c in X.columns if X[c].dtype == object]
    num_cols = [c for c in X.columns if c not in cat_cols]
    for c in num_cols:
        X[c] = pd.to_numeric(X[c], errors="coerce")
        if protocol == "impute":
            X[c] = X[c].fillna(X[c].median())
    for c in cat_cols:
        X[c] = X[c].astype("object")
        if protocol == "impute":
            X[c] = X[c].fillna("Unknown")
    X = pd.get_dummies(X, drop_first=True)
    X.columns = make_unique(X.columns)
    return X.astype(float), y.reset_index(drop=True), raw

def load_credit_approval():
    X, y, raw = process_credit("impute")
    return X, y, {
        "dataset": "credit_approval",
        "raw_missing_cells": int(raw.drop(columns=["target"]).isna().sum().sum()),
        "source": "UCI Australian Credit Approval",
    }

def load_har():
    mirrors = [
        "https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip",
        "http://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip",
    ]
    by = _download_bytes(mirrors)
    zf = zipfile.ZipFile(io.BytesIO(by))
    root = "UCI HAR Dataset/"
    with zf.open(root + "features.txt") as f:
        feats = pd.read_csv(f, sep=r"\s+", header=None, names=["idx", "name"])
    names = make_unique(feats["name"].astype(str))
    with zf.open(root + "train/X_train.txt") as f:
        Xtr = np.loadtxt(f)
    with zf.open(root + "test/X_test.txt") as f:
        Xte = np.loadtxt(f)
    with zf.open(root + "train/y_train.txt") as f:
        ytr = np.loadtxt(f).astype(int).ravel()
    with zf.open(root + "test/y_test.txt") as f:
        yte = np.loadtxt(f).astype(int).ravel()
    X = pd.DataFrame(np.vstack([Xtr, Xte]), columns=names)
    y = pd.Series(np.hstack([ytr, yte]) - 1, name="target")
    return X, y, {
        "dataset": "har",
        "raw_missing_cells": int(X.isna().sum().sum()),
        "source": "UCI HAR",
    }

def load_dataset(name):
    if name == "breast_cancer":
        return load_breast_cancer_dataset()
    if name == "credit_approval":
        return load_credit_approval()
    if name == "har":
        return load_har()
    raise ValueError(name)

def stratified_cap(X, y, max_n=None, seed=42):
    X, y = np.asarray(X, float), np.asarray(y, int)
    if max_n is None or len(y) <= max_n:
        return X, y
    splitter = StratifiedShuffleSplit(n_splits=1, train_size=max_n, random_state=seed)
    idx, _ = next(splitter.split(X, y))
    return X[idx], y[idx]

def standardized_dataset(name, seed=42):
    Xdf, ys, meta = load_dataset(name)
    X = StandardScaler().fit_transform(Xdf.to_numpy(float))
    y = ys.to_numpy(int)
    X, y = stratified_cap(X, y, MAX_N_PER_DATASET.get(name), seed)
    return X, y, meta


In [6]:
# %% [stress generation with observable clean counterpart and true irrelevant decoys]
def inject_outliers_with_mask(X, rate=0.10, scale=8.0, feature_frac=0.15, seed=42):
    rng = np.random.default_rng(seed)
    X = np.asarray(X, float).copy()
    n, d = X.shape
    row_mask = np.zeros(n, dtype=bool)
    cell_mask = np.zeros((n, d), dtype=bool)
    if rate <= 0:
        return X, row_mask, cell_mask
    m = max(1, int(round(rate * n)))
    rows = rng.choice(n, size=m, replace=False)
    row_mask[rows] = True
    q = max(1, int(round(feature_frac * d)))
    for i in rows:
        cols = rng.choice(d, size=q, replace=False)
        cell_mask[i, cols] = True
        perturb = np.clip(rng.standard_t(df=2, size=q) * scale, -5 * scale, 5 * scale)
        X[i, cols] += perturb
    return X, row_mask, cell_mask

def make_clone_spec(X_clean, clone_frac=0.50, clone_noise=0.01, max_clones=100, seed=42):
    X_clean = np.asarray(X_clean, float)
    n, d = X_clean.shape
    if clone_frac <= 0:
        return {"n_clones": 0, "source": np.array([], int), "aux": np.array([], int),
                "noise": np.empty((n, 0)), "clone_noise": clone_noise}
    rng = np.random.default_rng(seed)
    q = max(1, min(int(round(clone_frac * d)), max_clones))
    source = rng.choice(d, size=q, replace=True)
    aux = rng.choice(d, size=q, replace=True)
    noise = np.empty((n, q))
    for j, s in enumerate(source):
        noise[:, j] = rng.normal(0, clone_noise * (np.std(X_clean[:, s]) + 1e-12), size=n)
    return {"n_clones": q, "source": source, "aux": aux, "noise": noise, "clone_noise": clone_noise}

def apply_clone_spec(X, spec):
    X = np.asarray(X, float)
    if spec["n_clones"] == 0:
        return X.copy()
    clones = np.column_stack([
        0.95 * X[:, s] + 0.05 * X[:, t] + spec["noise"][:, j]
        for j, (s, t) in enumerate(zip(spec["source"], spec["aux"]))
    ])
    return np.column_stack([X, clones])

def build_stress_pair(X_clean, outlier_rate, clone_frac, seed):
    X_out, row_mask, cell_mask = inject_outliers_with_mask(
        X_clean, rate=outlier_rate, seed=seed
    )
    spec = make_clone_spec(
        X_clean, clone_frac=clone_frac, max_clones=MAX_CLONES, seed=seed + 1000
    )
    X_clean_aug = apply_clone_spec(X_clean, spec)
    X_stress_aug = apply_clone_spec(X_out, spec)
    d0 = X_clean.shape[1]
    clone_idx = list(range(d0, X_clean_aug.shape[1]))
    info = {
        "outlier_rate": float(outlier_rate),
        "clone_frac": float(clone_frac),
        "n_outlier_rows": int(row_mask.sum()),
        "n_clones": int(spec["n_clones"]),
        "d_original": int(d0),
        "d_total": int(X_clean_aug.shape[1]),
    }
    return X_clean_aug, X_stress_aug, row_mask, cell_mask, clone_idx, info

def append_irrelevant_decoys(X, n_decoys=3, seed=42):
    """Append distribution-matched but target-irrelevant columns.

    Each decoy is a row permutation of an existing column plus tiny noise. The marginal
    distribution is retained while its pairing with Y is broken. This is distinct from
    the near-collinear clone features used to induce multicollinearity.
    """
    rng = np.random.default_rng(seed)
    X = np.asarray(X, float)
    n, d = X.shape
    decoys = []
    for _ in range(n_decoys):
        src = int(rng.integers(0, d))
        col = rng.permutation(X[:, src]).copy()
        col += rng.normal(0, 0.01 * (np.std(col) + 1e-12), size=n)
        decoys.append(col)
    Xd = np.column_stack([X, np.column_stack(decoys)])
    decoy_idx = list(range(d, d + n_decoys))
    return Xd, decoy_idx


In [7]:

# %% [black-box learners]
def make_learner(kind, seed=42):
    if kind == "lgbm":
        if not LGBM_AVAILABLE:
            if QUICK_TEST:
                return HistGradientBoostingClassifier(max_iter=150, learning_rate=0.05, random_state=seed)
            raise RuntimeError("Full experiment requires LightGBM.")
        return LGBMClassifier(
            n_estimators=250, learning_rate=0.05, max_depth=10,
            subsample=0.9, colsample_bytree=0.9, random_state=seed,
            verbosity=-1, n_jobs=-1,
        )
    if kind == "svm":
        return make_pipeline(
            StandardScaler(),
            SVC(C=10.0, kernel="rbf", gamma="scale", probability=True, random_state=seed),
        )
    if kind == "mlp":
        return make_pipeline(
            StandardScaler(),
            MLPClassifier(
                hidden_layer_sizes=(100, 50), activation="relu",
                learning_rate_init=1e-3, alpha=1e-4, max_iter=250,
                early_stopping=True, random_state=seed,
            ),
        )
    raise ValueError(kind)

def model_classes(model):
    if hasattr(model, "classes_"):
        return np.asarray(model.classes_)
    if hasattr(model, "named_steps"):
        last = list(model.named_steps.values())[-1]
        if hasattr(last, "classes_"):
            return np.asarray(last.classes_)
    return None

def fit_clean_blackbox(X, y, learner, seed):
    idx = np.arange(len(y))
    train_idx, exp_idx = train_test_split(
        idx, test_size=0.30, random_state=seed, stratify=y
    )
    model = make_learner(learner, seed)
    t0 = time.perf_counter()
    model.fit(X[train_idx], y[train_idx])
    fit_time = time.perf_counter() - t0
    Y_exp = model.predict_proba(X[exp_idx])
    pred_idx = np.argmax(Y_exp, axis=1)
    classes = model_classes(model)
    pred = classes[pred_idx] if classes is not None else pred_idx
    acc = accuracy_score(y[exp_idx], pred)
    try:
        ll = log_loss(y[exp_idx], Y_exp, labels=np.unique(y))
    except Exception:
        ll = np.nan
    return model, train_idx, exp_idx, Y_exp, {
        "model_fit_time_sec": fit_time,
        "model_test_accuracy": float(acc),
        "model_test_log_loss": float(ll),
    }

def split_explanation_rows(y_exp, seed, fit_fraction=0.70):
    idx = np.arange(len(y_exp))
    fit_idx, val_idx = train_test_split(
        idx, train_size=fit_fraction, random_state=seed, stratify=y_exp
    )
    return np.asarray(fit_idx), np.asarray(val_idx)


In [8]:
# %% [inverse-map solver, decoy diagnostics, and stability]
AIME_METHODS = ["AIME", "HuberAIME", "RidgeAIME", "HuberRidgeAIME"]

def huber_weights(residual, delta=1.0):
    residual = np.asarray(residual, float)
    w = np.ones_like(residual)
    mask = residual > delta
    w[mask] = delta / np.maximum(residual[mask], 1e-12)
    return np.clip(w, 1e-8, 1.0)

def symmetric_condition(G):
    G = np.asarray(G, float)
    evals = np.linalg.eigvalsh((G + G.T) / 2)
    lam_min, lam_max = float(evals.min()), float(evals.max())
    cond = float(lam_max / max(lam_min, 1e-12))
    return cond, lam_min, lam_max

def svd_solve(G, rhs, rcond=1e-12):
    """SVD-based solve used for all inverse-map normal systems."""
    U, s, Vt = np.linalg.svd(np.asarray(G, float), full_matrices=False)
    tol = rcond * max(float(s.max()), 1e-300)
    inv = np.where(s > tol, 1.0 / s, 0.0)
    return (Vt.T * inv) @ (U.T @ np.asarray(rhs, float)), s, tol

def fit_inverse_map(
    X, Y, method="AIME", delta=1.0, ridge_lambda=1e-2,
    residual_mode="rms", max_iter=100, tol=1e-8
):
    """Fit A in X ≈ Y A^T using the manuscript equations."""
    X, Y = np.asarray(X, float), np.asarray(Y, float)
    if X.ndim != 2 or Y.ndim != 2 or len(X) != len(Y):
        raise ValueError(f"Expected X(n,d) and Y(n,C); got {X.shape} and {Y.shape}")
    n, d = X.shape
    C = Y.shape[1]
    use_huber = method in {"HuberAIME", "HuberRidgeAIME"}
    use_ridge = method in {"RidgeAIME", "HuberRidgeAIME"}
    lam = float(ridge_lambda if use_ridge else 0.0)

    w = np.ones(n)
    B = np.zeros((C, d))  # B = A^T
    converged = not use_huber
    n_iter = 1

    for it in range(max_iter if use_huber else 1):
        G = Y.T @ (Y * w[:, None])
        G_reg = G + lam * np.eye(C)
        rhs = Y.T @ (X * w[:, None])
        B_new, _, _ = svd_solve(G_reg, rhs)
        residual_norm = np.linalg.norm(X - Y @ B_new, axis=1)
        residual_score = residual_norm / np.sqrt(max(1, d)) if residual_mode == "rms" else residual_norm

        if not use_huber:
            B = B_new
            break

        w_new = huber_weights(residual_score, delta)
        n_iter = it + 1
        coef_change = np.linalg.norm(B_new - B) / (np.linalg.norm(B) + 1e-12)
        weight_change = np.max(np.abs(w_new - w))
        B, w = B_new, w_new
        if coef_change <= tol and weight_change <= np.sqrt(tol):
            converged = True
            break
    else:
        converged = False

    # Final SVD solve at the final IRLS weights, so B and the reported system agree exactly.
    G = Y.T @ (Y * w[:, None])
    G_reg = G + lam * np.eye(C)
    rhs = Y.T @ (X * w[:, None])
    B, system_singular_values, solver_tol = svd_solve(G_reg, rhs)
    residual_norm = np.linalg.norm(X - Y @ B, axis=1)
    residual_score = residual_norm / np.sqrt(max(1, d)) if residual_mode == "rms" else residual_norm

    cond_raw, min_raw, max_raw = symmetric_condition(G)
    cond_reg, min_reg, max_reg = symmetric_condition(G_reg)
    A = B.T
    if A.shape != (d, C):
        raise AssertionError(f"Inverse orientation failure: A has shape {A.shape}, expected {(d,C)}")

    diag = {
        "orientation": "X ~= Y A^T",
        "solver": "SVD normal-system solve",
        "cond_raw": cond_raw,
        "cond_reg": cond_reg,
        "lambda_min_raw": min_raw,
        "lambda_min_reg": min_reg,
        "lambda_max_raw": max_raw,
        "lambda_max_reg": max_reg,
        "system_min_singular_value": float(system_singular_values.min()),
        "solver_svd_tolerance": float(solver_tol),
        "coef_fro_norm": float(np.linalg.norm(A, "fro")),
        "mean_iter": int(n_iter),
        "converged": bool(converged),
        "mean_huber_weight": float(w.mean()),
        "frac_downweighted": float(np.mean(w < 0.999)),
        "ridge_lambda": lam,
        "huber_delta": float(delta),
        "residual_mode": residual_mode,
        "effective_delta_on_l2_norm": float(delta * np.sqrt(d)) if residual_mode == "rms" else float(delta),
    }
    state = {
        "A": A, "B": B, "weights": w, "residual_score": residual_score,
        "method": method, "ridge_lambda": lam, "huber_delta": float(delta),
        "residual_mode": residual_mode,
    }
    return A, diag, state

def inverse_reconstruction_metrics(state, Y, X_target, prefix=""):
    X_target = np.asarray(X_target, float)
    X_hat = np.asarray(Y, float) @ state["B"]
    err = X_target - X_hat
    mse = float(np.mean(err ** 2))
    rmse = float(np.sqrt(mse))
    denom = float(np.sum((X_target - X_target.mean(axis=0, keepdims=True)) ** 2))
    r2 = float(1 - np.sum(err ** 2) / (denom + 1e-12))
    cos = cosine_safe(X_target, X_hat)
    return {
        f"{prefix}reconstruction_mse": mse,
        f"{prefix}reconstruction_rmse": rmse,
        f"{prefix}reconstruction_r2": r2,
        f"{prefix}reconstruction_cosine": cos,
    }

def operator_recovery_metrics(A, A_reference, prefix="operator_"):
    g, g_ref = global_strength(A), global_strength(A_reference)
    return {
        f"{prefix}cosine_flat": cosine_safe(A, A_reference),
        f"{prefix}spearman_global": spearman_safe(g, g_ref),
        f"{prefix}topk_jaccard": topk_jaccard(g, g_ref, TOP_K),
    }

def subset_mass_metrics(A, feature_idx, prefix):
    g = global_strength(A)
    if not feature_idx:
        return {f"{prefix}_mass_ratio": 0.0, f"{prefix}_topk_infiltration": 0.0}
    mass = float(g[feature_idx].sum() / (g.sum() + 1e-12))
    top = topk_set(g, TOP_K)
    infiltration = float(sum(i in top for i in feature_idx) / len(feature_idx))
    return {f"{prefix}_mass_ratio": mass, f"{prefix}_topk_infiltration": infiltration}

def outlier_weight_metrics(state, known_outlier_mask):
    mask = np.asarray(known_outlier_mask, bool)
    weights = np.asarray(state["weights"], float)
    if len(mask) != len(weights) or mask.sum() == 0 or (~mask).sum() == 0:
        return {
            "outlier_weight_ap": np.nan,
            "mean_weight_outlier_rows": np.nan,
            "mean_weight_nonoutlier_rows": np.nan,
        }
    return {
        "outlier_weight_ap": float(average_precision_score(mask.astype(int), 1 - weights)),
        "mean_weight_outlier_rows": float(weights[mask].mean()),
        "mean_weight_nonoutlier_rows": float(weights[~mask].mean()),
    }

def bootstrap_inverse_stability(X, Y, method, delta, lam, A0=None, B=10, seed=42):
    rng = np.random.default_rng(seed)
    if A0 is None:
        A0, _, _ = fit_inverse_map(X, Y, method, delta, lam, RESIDUAL_SCALE_MODE)
    vals = []
    n = len(X)
    for _ in range(B):
        idx = rng.choice(n, size=n, replace=True)
        Ab, _, _ = fit_inverse_map(X[idx], Y[idx], method, delta, lam, RESIDUAL_SCALE_MODE)
        vals.append({
            "bootstrap_cosine_flat": cosine_safe(A0, Ab),
            "bootstrap_spearman_global": spearman_safe(global_strength(A0), global_strength(Ab)),
            "bootstrap_topk_jaccard": topk_jaccard(global_strength(A0), global_strength(Ab), TOP_K),
        })
    return pd.DataFrame(vals).mean().to_dict()

def irrelevant_decoy_metrics(X, Y, method, delta, lam, trials=5, seed=42):
    rows = []
    for t in range(trials):
        Xd, decoy_idx = append_irrelevant_decoys(X, IRRELEVANT_DECOY_COUNT, seed + 1009 * t)
        Ad, _, _ = fit_inverse_map(Xd, Y, method, delta, lam, RESIDUAL_SCALE_MODE)
        rows.append(subset_mass_metrics(Ad, decoy_idx, "irrelevant_decoy"))
    return pd.DataFrame(rows).mean().to_dict()


## Validation unit tests for the corrected inverse orientation

In [9]:

# %% [inverse-map implementation unit tests]
def run_inverse_map_unit_tests():
    rng = np.random.default_rng(MAIN_SEED)
    n, d, C = 160, 24, 3
    Y = softmax(rng.normal(size=(n, C)), axis=1)
    B_true = rng.normal(size=(C, d))
    X_clean = Y @ B_true + 0.02 * rng.normal(size=(n, d))

    # AIME must match the direct Moore–Penrose least-squares solution B = pinv(Y) X.
    A, _, _ = fit_inverse_map(X_clean, Y, "AIME", DEFAULT_HUBER_DELTA, DEFAULT_RIDGE_LAMBDA, RESIDUAL_SCALE_MODE)
    B_direct = np.linalg.pinv(Y) @ X_clean
    err_aime = float(np.max(np.abs(A.T - B_direct)))

    # With a very large Huber threshold, HRA must reduce to RidgeAIME.
    Ar, _, _ = fit_inverse_map(X_clean, Y, "RidgeAIME", 1e9, DEFAULT_RIDGE_LAMBDA, RESIDUAL_SCALE_MODE)
    Ah, _, _ = fit_inverse_map(X_clean, Y, "HuberRidgeAIME", 1e9, DEFAULT_RIDGE_LAMBDA, RESIDUAL_SCALE_MODE)
    err_hra_ridge = float(np.max(np.abs(Ar - Ah)))

    # Injected input-space outlier rows must receive lower Huber weights on average.
    X_out, mask, _ = inject_outliers_with_mask(X_clean, rate=0.10, scale=8.0, seed=MAIN_SEED + 1)
    _, _, state = fit_inverse_map(X_out, Y, "HuberRidgeAIME", DEFAULT_HUBER_DELTA, DEFAULT_RIDGE_LAMBDA, RESIDUAL_SCALE_MODE)
    w_out = float(state["weights"][mask].mean())
    w_in = float(state["weights"][~mask].mean())

    result = pd.DataFrame([{
        "aime_max_abs_error_vs_pinv": err_aime,
        "hra_max_abs_error_vs_ridge_at_large_delta": err_hra_ridge,
        "mean_weight_outlier_rows": w_out,
        "mean_weight_nonoutlier_rows": w_in,
        "aime_orientation_pass": err_aime < 1e-8,
        "large_delta_equivalence_pass": err_hra_ridge < 1e-8,
        "outlier_downweight_pass": w_out < w_in,
    }])
    save_csv(result, MAIN_DATADIR / "main_corrected_inverse_map_unit_tests.csv")
    if not bool(result[["aime_orientation_pass", "large_delta_equivalence_pass", "outlier_downweight_pass"]].all(axis=None)):
        raise AssertionError("Corrected inverse-map unit tests failed.")
    print("All inverse-map unit tests passed.")
    return result

MAIN_UNIT_TESTS = run_inverse_map_unit_tests()
display(MAIN_UNIT_TESTS)


[csv] /Users/takafumi/Documents/Python/HuberRidgeAIME/notebooks/output/main_repro_corrected/data/main_corrected_inverse_map_unit_tests.csv
All inverse-map unit tests passed.


,aime_max_abs_error_vs_pinv,hra_max_abs_error_vs_ridge_at_large_delta,mean_weight_outlier_rows,mean_weight_nonoutlier_rows,aime_orientation_pass,large_delta_equivalence_pass,outlier_downweight_pass
0,4.329870e-15,0.0,0.310595,1.0,True,True,True


## 1. Dataset validity, missingness, rank, correlation, and heavy-tail audit

In [10]:

# %% [dataset diagnostics for Reviewer 5 minor points]
def effective_rank(s):
    s = np.asarray(s, float)
    s = s[s > 1e-12]
    if not len(s):
        return 0.0
    p = s / s.sum()
    return float(np.exp(-(p * np.log(p + 1e-15)).sum()))

def dataset_diagnostics(X):
    Xs = StandardScaler().fit_transform(np.asarray(X, float))
    n, d = Xs.shape
    s = np.linalg.svd(Xs, compute_uv=False, full_matrices=False)
    cond_xtx = float((s[0] / max(s[-1], 1e-12)) ** 2)
    C = np.nan_to_num(np.corrcoef(Xs, rowvar=False), nan=0.0)
    iu = np.triu_indices_from(C, 1)
    abs_corr = np.abs(C[iu])
    med = np.median(Xs, axis=0)
    mad = np.median(np.abs(Xs - med), axis=0)
    rz = 0.6745 * (Xs - med) / np.where(mad < 1e-12, 1.0, mad)
    kurt = stats.kurtosis(Xs, axis=0, fisher=True, bias=False, nan_policy="omit")
    return {
        "n": n, "d": d,
        "matrix_rank": int(np.linalg.matrix_rank(Xs)),
        "effective_rank_ratio": effective_rank(s) / max(1, d),
        "near_zero_singular_values": int(np.sum(s <= 1e-10 * max(s[0], 1e-12))),
        "log10_condition_XtX": float(np.log10(max(cond_xtx, 1e-300))),
        "median_abs_correlation": float(np.median(abs_corr)),
        "frac_abs_corr_gt_0.50": float(np.mean(abs_corr > 0.50)),
        "frac_abs_corr_gt_0.70": float(np.mean(abs_corr > 0.70)),
        "frac_abs_corr_gt_0.90": float(np.mean(abs_corr > 0.90)),
        "robust_outlier_cell_rate": float(np.mean(np.abs(rz) > 3.5)),
        "median_excess_kurtosis": float(np.nanmedian(kurt)),
        "q90_excess_kurtosis": float(np.nanquantile(kurt, 0.90)),
        "frac_features_excess_kurtosis_gt_10": float(np.nanmean(kurt > 10)),
    }


def run_main_dataset_audit():
    out = MAIN_DATADIR / "main_corrected_dataset_audit.csv"
    cached = load_versioned_csv(out)
    if cached is not None:
        return cached
    rows = []
    audit_datasets = DATASETS if QUICK_TEST else ["breast_cancer", "credit_approval", "har"]
    for dataset in audit_datasets:
        Xdf, y, meta = load_dataset(dataset)
        diag = dataset_diagnostics(Xdf.to_numpy(float))
        counts = pd.Series(y).value_counts().sort_index().to_dict()
        rows.append({
            "dataset": dataset,
            "raw_missing_cells": meta["raw_missing_cells"],
            "n_classes": int(pd.Series(y).nunique()),
            "class_counts": "; ".join(f"{k}:{v}" for k, v in counts.items()),
            **diag,
        })
    df = pd.DataFrame(rows)
    save_table_bundle(
        df,
        "table_main_corrected_dataset_audit",
        "Dataset validity, missingness, rank, multicollinearity, robust-outlier, and heavy-tail diagnostics after numerical encoding. Pairwise correlation fractions and singular-spectrum diagnostics are reported together because a high condition number need not imply many pairwise correlations above 0.90.",
        "tab:main_corrected_dataset_audit",
    )
    return df

MAIN_DATASET_AUDIT = run_main_dataset_audit()
display(MAIN_DATASET_AUDIT)


[csv] /Users/takafumi/Documents/Python/HuberRidgeAIME/notebooks/output/main_repro_corrected/data/table_main_corrected_dataset_audit.csv
[tex] /Users/takafumi/Documents/Python/HuberRidgeAIME/notebooks/output/main_repro_corrected/tables/table_main_corrected_dataset_audit.tex


,dataset,raw_missing_cells,n_classes,class_counts,n,d,matrix_rank,effective_rank_ratio,near_zero_singular_values,log10_condition_XtX,median_abs_correlation,frac_abs_corr_gt_0.50,frac_abs_corr_gt_0.70,frac_abs_corr_gt_0.90,robust_outlier_cell_rate,median_excess_kurtosis,q90_excess_kurtosis,frac_features_excess_kurtosis_gt_10
0,breast_cancer,0,2,0:212; 1:357,569,30,30,0.558777,0,4.999253,0.345007,0.333333,0.160920,0.048276,0.029818,3.022590,21.889799,0.200000
1,credit_approval,67,2,0:383; 1:307,690,42,38,0.801707,4,27.472007,0.040543,0.019744,0.011614,0.010453,0.017219,8.049270,210.076450,0.452381
2,har,0,6,0:1722; 1:1544; 2:1406; 3:1777; 4:1906; 5:1944,10299,561,540,0.366865,21,30.467093,0.379293,0.413032,0.230768,0.051522,0.234115,0.586898,36.072109,0.244207


## 2. Clean AIME-family benchmark under the corrected inverse orientation

In [11]:

# %% [clean benchmark]
def run_clean_benchmark():
    out = MAIN_DATADIR / "main_corrected_clean_benchmark_raw.csv"
    cached = load_versioned_csv(out)
    if cached is not None:
        return cached
    if not RUN_CLEAN_BENCHMARK:
        return pd.DataFrame()
    rows = []
    total = len(DATASETS) * len(LEARNERS) * CLEAN_REPEATS
    job = 0
    for dataset in DATASETS:
        X, y, _ = standardized_dataset(dataset, MAIN_SEED)
        for rep in range(CLEAN_REPEATS):
            for learner in LEARNERS:
                job += 1
                seed = MAIN_SEED + 10000 * rep + 100 * (LEARNERS.index(learner) + 1)
                print(f"[clean {job}/{total}] {dataset} {learner} rep={rep}")
                try:
                    model, _, exp_idx, Y_exp, model_info = fit_clean_blackbox(X, y, learner, seed)
                    X_exp, y_exp = X[exp_idx], y[exp_idx]
                    fit_idx, val_idx = split_explanation_rows(y_exp, seed + 17)
                    for method in AIME_METHODS:
                        t0 = time.perf_counter()
                        A, diag, state = fit_inverse_map(
                            X_exp[fit_idx], Y_exp[fit_idx], method,
                            DEFAULT_HUBER_DELTA, DEFAULT_RIDGE_LAMBDA, RESIDUAL_SCALE_MODE,
                        )
                        elapsed = time.perf_counter() - t0
                        rec = inverse_reconstruction_metrics(state, Y_exp[val_idx], X_exp[val_idx], "clean_target_")
                        boot = bootstrap_inverse_stability(
                            X_exp[fit_idx], Y_exp[fit_idx], method,
                            DEFAULT_HUBER_DELTA, DEFAULT_RIDGE_LAMBDA,
                            A0=A, B=CLEAN_BOOTSTRAPS, seed=seed + 29,
                        )
                        decoy = irrelevant_decoy_metrics(
                            X_exp[fit_idx], Y_exp[fit_idx], method,
                            DEFAULT_HUBER_DELTA, DEFAULT_RIDGE_LAMBDA,
                            CLEAN_DECOY_TRIALS, seed + 61,
                        )
                        rows.append({
                            "dataset": dataset, "learner": learner, "repeat": rep,
                            "method": method, **model_info, "explain_time_sec": elapsed,
                            "n_explanation_fit": len(fit_idx), "n_explanation_val": len(val_idx),
                            **diag, **rec, **boot, **decoy, "error": "",
                        })
                except Exception as e:
                    rows.append({"dataset": dataset, "learner": learner, "repeat": rep, "method": "FAILED", "error": str(e)})
    df = pd.DataFrame(rows)
    if len(df):
        df["log10_cond_raw"] = np.log10(df["cond_raw"].clip(lower=1e-300))
        df["log10_cond_reg"] = np.log10(df["cond_reg"].clip(lower=1e-300))
        df["log10_coef_norm"] = np.log10(df["coef_fro_norm"].clip(lower=1e-300))
        save_csv(df, out)
    return df

def summarize_clean_benchmark(df):
    ok = df[df["error"].fillna("") == ""].copy()
    metrics = [
        "model_test_accuracy", "clean_target_reconstruction_r2",
        "clean_target_reconstruction_cosine", "bootstrap_cosine_flat",
        "bootstrap_spearman_global", "irrelevant_decoy_mass_ratio",
        "irrelevant_decoy_topk_infiltration", "log10_cond_reg",
        "log10_coef_norm", "explain_time_sec", "mean_iter",
    ]
    summary = ok.groupby("method")[metrics].agg(["mean", "std", "median", "count"]).reset_index()
    save_table_bundle(
        summary, "table_main_corrected_clean_benchmark_summary",
        "Clean-benchmark AIME-family results under the corrected inverse orientation X approximately equals Y A transpose. Coefficient norm is a regularization diagnostic only, not an explanation-quality score.",
        "tab:main_corrected_clean_benchmark",
    )
    return summary

MAIN_CLEAN_RAW = run_clean_benchmark()
MAIN_CLEAN_SUMMARY = summarize_clean_benchmark(MAIN_CLEAN_RAW) if len(MAIN_CLEAN_RAW) else pd.DataFrame()
display(MAIN_CLEAN_SUMMARY)


[clean 1/27] breast_cancer lgbm rep=0
[clean 2/27] breast_cancer svm rep=0
[clean 3/27] breast_cancer mlp rep=0
[clean 4/27] breast_cancer lgbm rep=1
[clean 5/27] breast_cancer svm rep=1
[clean 6/27] breast_cancer mlp rep=1
[clean 7/27] breast_cancer lgbm rep=2
[clean 8/27] breast_cancer svm rep=2
[clean 9/27] breast_cancer mlp rep=2
[clean 10/27] credit_approval lgbm rep=0
[clean 11/27] credit_approval svm rep=0
[clean 12/27] credit_approval mlp rep=0
[clean 13/27] credit_approval lgbm rep=1
[clean 14/27] credit_approval svm rep=1
[clean 15/27] credit_approval mlp rep=1
[clean 16/27] credit_approval lgbm rep=2
[clean 17/27] credit_approval svm rep=2
[clean 18/27] credit_approval mlp rep=2
[clean 19/27] har lgbm rep=0
[clean 20/27] har svm rep=0
[clean 21/27] har mlp rep=0
[clean 22/27] har lgbm rep=1
[clean 23/27] har svm rep=1
[clean 24/27] har mlp rep=1
[clean 25/27] har lgbm rep=2
[clean 26/27] har svm rep=2
[clean 27/27] har mlp rep=2
[csv] /Users/takafumi/Documents/Python/HuberRi

method model_test_accuracy                            \
                                 mean       std    median count   
0            AIME            0.921455  0.063541  0.956667    27   
1       HuberAIME            0.921455  0.063541  0.956667    27   
2  HuberRidgeAIME            0.921455  0.063541  0.956667    27   
3       RidgeAIME            0.921455  0.063541  0.956667    27   

  clean_target_reconstruction_r2                            \
                            mean       std    median count   
0                       0.283265  0.202223  0.305853    27   
1                       0.283745  0.200946  0.307046    27   
2                       0.283748  0.200944  0.307055    27   
3                       0.283270  0.202221  0.305867    27   

  clean_target_reconstruction_cosine  ... log10_coef_norm        \
                                mean  ...          median count   
0                           0.497539  ...        0.702441    27   
1                           0.498297  ...        0.689489    27   
2                           0.498295  ...        0.689383    27   
3                           0.497537  ...        0.702346    27   

  explain_time_sec                           mean_iter                         
              mean       std    median count      mean       std median count  
0         0.000637  0.000597  0.000252    27  1.000000  0.000000    1.0    27  
1         0.002098  0.002191  0.000596    27  7.592593  0.797074    7.0    27  
2         0.002075  0.002178  0.000592    27  7.592593  0.797074    7.0    27  
3         0.000572  0.000610  0.000158    27  1.000000  0.000000    1.0    27  

[4 rows x 45 columns]

## 3. Controlled real-data stress benchmark for all four AIME-family estimators

In [12]:

# %% [all-method end-to-end controlled stress]
def collect_stress_method_row(
    *, dataset, learner, repeat, method, stress_info, model_info,
    X_fit, Y_fit, X_val_observed, Y_val_observed, X_val_clean, Y_val_clean,
    A_reference, clone_idx, outlier_mask_fit, seed
):
    t0 = time.perf_counter()
    A, diag, state = fit_inverse_map(
        X_fit, Y_fit, method, DEFAULT_HUBER_DELTA, DEFAULT_RIDGE_LAMBDA, RESIDUAL_SCALE_MODE
    )
    elapsed = time.perf_counter() - t0
    return {
        "dataset": dataset, "learner": learner, "repeat": repeat, "method": method,
        **stress_info, **model_info, "explain_time_sec": elapsed,
        "n_explanation_fit": len(X_fit), "n_explanation_val": len(X_val_observed),
        **diag,
        **inverse_reconstruction_metrics(state, Y_val_clean, X_val_clean, "clean_target_"),
        **inverse_reconstruction_metrics(state, Y_val_observed, X_val_observed, "observed_target_"),
        **operator_recovery_metrics(A, A_reference),
        **subset_mass_metrics(A, clone_idx, "clone"),
        **irrelevant_decoy_metrics(
            X_fit, Y_fit, method, DEFAULT_HUBER_DELTA, DEFAULT_RIDGE_LAMBDA,
            STRESS_DECOY_TRIALS, seed + 61,
        ),
        **outlier_weight_metrics(state, outlier_mask_fit),
        **bootstrap_inverse_stability(
            X_fit, Y_fit, method, DEFAULT_HUBER_DELTA, DEFAULT_RIDGE_LAMBDA,
            A0=A, B=STRESS_BOOTSTRAPS, seed=seed + 29,
        ),
        "error": "",
    }

def run_full_stress_benchmark():
    out = MAIN_DATADIR / "main_corrected_full_stress_raw.csv"
    cached = load_versioned_csv(out)
    if cached is not None:
        return cached
    if not RUN_FULL_STRESS:
        return pd.DataFrame()
    rows = []
    total = len(DATASETS) * len(LEARNERS) * STRESS_REPEATS * len(STRESS_OUTLIER_LEVELS) * len(STRESS_CLONE_FRACS)
    job = 0
    for dataset in DATASETS:
        X, y, _ = standardized_dataset(dataset, MAIN_SEED)
        for rep in range(STRESS_REPEATS):
            for outlier_rate in STRESS_OUTLIER_LEVELS:
                for clone_frac in STRESS_CLONE_FRACS:
                    seed0 = MAIN_SEED + rep * 10000 + int(outlier_rate * 1000) + int(clone_frac * 100)
                    X_clean_aug, X_stress_aug, out_mask, _, clone_idx, stress_info = build_stress_pair(
                        X, outlier_rate, clone_frac, seed0
                    )
                    for learner in LEARNERS:
                        job += 1
                        seed = seed0 + 100 * (LEARNERS.index(learner) + 1)
                        print(f"[stress {job}/{total}] {dataset} {learner} rep={rep} pi={outlier_rate} gamma={clone_frac}")
                        try:
                            train_idx, test_idx = train_test_split(
                                np.arange(len(y)), test_size=0.30, random_state=seed, stratify=y
                            )
                            model = make_learner(learner, seed)
                            t0 = time.perf_counter()
                            model.fit(X_stress_aug[train_idx], y[train_idx])
                            model_fit_time = time.perf_counter() - t0
                            Y_stress = model.predict_proba(X_stress_aug[test_idx])
                            Y_clean = model.predict_proba(X_clean_aug[test_idx])
                            classes = model_classes(model)
                            pred_idx = np.argmax(Y_stress, axis=1)
                            pred = classes[pred_idx] if classes is not None else pred_idx
                            model_info = {
                                "model_fit_time_sec": model_fit_time,
                                "model_test_accuracy": float(accuracy_score(y[test_idx], pred)),
                                "model_test_log_loss": float(log_loss(y[test_idx], Y_stress, labels=np.unique(y))),
                            }
                            fit_idx, val_idx = split_explanation_rows(y[test_idx], seed + 17)
                            X_clean_test, X_stress_test = X_clean_aug[test_idx], X_stress_aug[test_idx]
                            out_mask_test = out_mask[test_idx]
                            for method in AIME_METHODS:
                                A_ref, _, _ = fit_inverse_map(
                                    X_clean_test[fit_idx], Y_clean[fit_idx], method,
                                    DEFAULT_HUBER_DELTA, DEFAULT_RIDGE_LAMBDA, RESIDUAL_SCALE_MODE,
                                )
                                rows.append(collect_stress_method_row(
                                    dataset=dataset, learner=learner, repeat=rep, method=method,
                                    stress_info=stress_info, model_info=model_info,
                                    X_fit=X_stress_test[fit_idx], Y_fit=Y_stress[fit_idx],
                                    X_val_observed=X_stress_test[val_idx], Y_val_observed=Y_stress[val_idx],
                                    X_val_clean=X_clean_test[val_idx], Y_val_clean=Y_clean[val_idx],
                                    A_reference=A_ref, clone_idx=clone_idx,
                                    outlier_mask_fit=out_mask_test[fit_idx], seed=seed,
                                ))
                        except Exception as e:
                            rows.append({
                                "dataset": dataset, "learner": learner, "repeat": rep,
                                "outlier_rate": outlier_rate, "clone_frac": clone_frac,
                                "method": "FAILED", "error": str(e),
                            })
    df = pd.DataFrame(rows)
    if len(df):
        df["log10_cond_raw"] = np.log10(df["cond_raw"].clip(lower=1e-300))
        df["log10_cond_reg"] = np.log10(df["cond_reg"].clip(lower=1e-300))
        df["log10_coef_norm"] = np.log10(df["coef_fro_norm"].clip(lower=1e-300))
        save_csv(df, out)
    return df

PAIRWISE_METRICS = [
    ("clean_target_reconstruction_r2", "higher is better", "clean-target reconstruction"),
    ("clean_target_reconstruction_cosine", "higher is better", "clean-target reconstruction"),
    ("operator_cosine_flat", "higher is better", "same-method clean-reference operator recovery"),
    ("operator_topk_jaccard", "higher is better", "same-method clean-reference operator recovery"),
    ("bootstrap_cosine_flat", "higher is better", "stability/self-consistency"),
    ("irrelevant_decoy_mass_ratio", "lower is better", "true irrelevant-decoy attribution"),
    ("clone_mass_ratio", "descriptive", "correlated-clone redistribution"),
    ("log10_cond_reg", "lower is better", "conditioning diagnostic"),
    ("log10_coef_norm", "diagnostic only", "regularization diagnostic"),
]
METHOD_CONTRASTS = [
    ("HuberRidgeAIME", "RidgeAIME", "Huber contribution beyond ridge"),
    ("HuberRidgeAIME", "HuberAIME", "ridge contribution beyond Huber"),
    ("RidgeAIME", "AIME", "ridge contribution"),
    ("HuberAIME", "AIME", "Huber contribution"),
]

def pairwise_stress_cluster_ci(df):
    ok = df[df["error"].fillna("") == ""].copy()
    pair_key = ["dataset", "learner", "outlier_rate", "clone_frac", "repeat"]
    cluster_key = ["dataset", "learner", "outlier_rate", "clone_frac"]
    rows = []
    subsets = [("all_conditions", ok), ("outlier_present", ok[ok["outlier_rate"] > 0])]
    for rate in sorted(ok["outlier_rate"].unique()):
        subsets.append((f"outlier_rate={rate:.2f}", ok[np.isclose(ok["outlier_rate"], rate)]))
    subsets.append(("joint_high_stress", ok[
        np.isclose(ok["outlier_rate"], max(STRESS_OUTLIER_LEVELS)) &
        np.isclose(ok["clone_frac"], max(STRESS_CLONE_FRACS))
    ]))
    for subset_name, sub in subsets:
        for method_a, method_b, interpretation in METHOD_CONTRASTS:
            for metric, direction, role in PAIRWISE_METRICS:
                piv = sub.pivot_table(index=pair_key, columns="method", values=metric, aggfunc="mean")
                if not {method_a, method_b}.issubset(piv.columns):
                    continue
                piv = piv[[method_a, method_b]].dropna().reset_index()
                piv["difference"] = piv[method_a] - piv[method_b]
                cluster_diff = piv.groupby(cluster_key, dropna=False)["difference"].mean()
                ci = cluster_bootstrap_ci(cluster_diff.to_numpy(), CLUSTER_BOOTSTRAP_RESAMPLES, seed=MAIN_SEED + len(rows))
                _, p = wilcoxon_two_sided(cluster_diff.to_numpy())
                rows.append({
                    "subset": subset_name, "method_a": method_a, "method_b": method_b,
                    "contrast_interpretation": interpretation, "metric": metric,
                    "metric_role": role, "preferred_direction": direction,
                    "n_repeated_pairs": len(piv), "n_condition_clusters": ci["n_clusters"],
                    "method_a_mean": float(piv[method_a].mean()),
                    "method_b_mean": float(piv[method_b].mean()),
                    "a_minus_b_cluster_mean": ci["mean"],
                    "CI95_low": ci["ci_low"], "CI95_high": ci["ci_high"],
                    "median_cluster_difference": ci["median"],
                    "wilcoxon_p_on_cluster_means": p,
                })
    return pd.DataFrame(rows)

def summarize_stress(df):
    ok = df[df["error"].fillna("") == ""].copy()
    metrics = [
        "clean_target_reconstruction_r2", "clean_target_reconstruction_cosine",
        "operator_cosine_flat", "operator_topk_jaccard", "bootstrap_cosine_flat",
        "irrelevant_decoy_mass_ratio", "irrelevant_decoy_topk_infiltration",
        "clone_mass_ratio", "log10_cond_reg", "log10_coef_norm", "explain_time_sec",
    ]
    summary = ok.groupby(["method", "outlier_rate"])[metrics].agg(["mean", "std", "median", "count"]).reset_index()
    save_table_bundle(
        summary, "table_main_corrected_stress_summary",
        "Controlled real-data stress results for all AIME-family estimators under the corrected inverse orientation. Coefficient norm is diagnostic only. Correlated-clone mass and true irrelevant-decoy mass are reported separately.",
        "tab:main_corrected_stress_summary",
    )
    ci = pairwise_stress_cluster_ci(ok)
    save_table_bundle(
        ci, "table_main_corrected_pairwise_cluster_ci",
        "Paired method contrasts in the controlled real-data stress benchmark. Repeated seeds are averaged within dataset--learner--stress cells before condition-cluster bootstrap 95 percent confidence intervals.",
        "tab:main_corrected_pairwise_cluster_ci",
    )
    return summary, ci

MAIN_STRESS_RAW = run_full_stress_benchmark()
if len(MAIN_STRESS_RAW):
    MAIN_STRESS_SUMMARY, MAIN_PAIRWISE_CI = summarize_stress(MAIN_STRESS_RAW)
else:
    MAIN_STRESS_SUMMARY, MAIN_PAIRWISE_CI = pd.DataFrame(), pd.DataFrame()
display(MAIN_STRESS_SUMMARY)
display(MAIN_PAIRWISE_CI.head(30))


[stress 1/243] breast_cancer lgbm rep=0 pi=0.0 gamma=0.0
[stress 2/243] breast_cancer svm rep=0 pi=0.0 gamma=0.0
[stress 3/243] breast_cancer mlp rep=0 pi=0.0 gamma=0.0
[stress 4/243] breast_cancer lgbm rep=0 pi=0.0 gamma=0.25
[stress 5/243] breast_cancer svm rep=0 pi=0.0 gamma=0.25
[stress 6/243] breast_cancer mlp rep=0 pi=0.0 gamma=0.25
[stress 7/243] breast_cancer lgbm rep=0 pi=0.0 gamma=0.5
[stress 8/243] breast_cancer svm rep=0 pi=0.0 gamma=0.5
[stress 9/243] breast_cancer mlp rep=0 pi=0.0 gamma=0.5
[stress 10/243] breast_cancer lgbm rep=0 pi=0.05 gamma=0.0
[stress 11/243] breast_cancer svm rep=0 pi=0.05 gamma=0.0
[stress 12/243] breast_cancer mlp rep=0 pi=0.05 gamma=0.0
[stress 13/243] breast_cancer lgbm rep=0 pi=0.05 gamma=0.25
[stress 14/243] breast_cancer svm rep=0 pi=0.05 gamma=0.25
[stress 15/243] breast_cancer mlp rep=0 pi=0.05 gamma=0.25
[stress 16/243] breast_cancer lgbm rep=0 pi=0.05 gamma=0.5
[stress 17/243] breast_cancer svm rep=0 pi=0.05 gamma=0.5
[stress 18/243] brea

method outlier_rate clean_target_reconstruction_r2            \
                                                          mean       std   
0             AIME         0.00                       0.280404  0.201377   
1             AIME         0.05                       0.261507  0.201290   
2             AIME         0.10                       0.252926  0.209269   
3        HuberAIME         0.00                       0.280787  0.200128   
4        HuberAIME         0.05                       0.279397  0.196475   
5        HuberAIME         0.10                       0.287344  0.197757   
6   HuberRidgeAIME         0.00                       0.280788  0.200125   
7   HuberRidgeAIME         0.05                       0.279398  0.196470   
8   HuberRidgeAIME         0.10                       0.287345  0.197750   
9        RidgeAIME         0.00                       0.280408  0.201375   
10       RidgeAIME         0.05                       0.261519  0.201282   
11       RidgeAIME         0.10                       0.252951  0.209250   

                   clean_target_reconstruction_cosine                      \
      median count                               mean       std    median   
0   0.316599    81                           0.492726  0.220175  0.571662   
1   0.304906    81                           0.478589  0.220717  0.562019   
2   0.301136    81                           0.477072  0.220563  0.557053   
3   0.310457    81                           0.493292  0.218622  0.572723   
4   0.323739    81                           0.493735  0.214792  0.581659   
5   0.329471    81                           0.503081  0.210994  0.580952   
6   0.310427    81                           0.493291  0.218623  0.572722   
7   0.323722    81                           0.493732  0.214794  0.581658   
8   0.329443    81                           0.503078  0.210996  0.580948   
9   0.316582    81                           0.492725  0.220176  0.571656   
10  0.304932    81                           0.478585  0.220720  0.562015   
11  0.301110    81                           0.477067  0.220566  0.557048   

          ... log10_cond_reg       log10_coef_norm                            \
   count  ...         median count            mean       std    median count   
0     81  ...       0.219654    81        0.984677  0.493915  0.745938    81   
1     81  ...       0.231540    81        1.026423  0.471718  0.817633    81   
2     81  ...       0.242290    81        1.061918  0.451840  0.873819    81   
3     81  ...       0.232533    81        0.973246  0.495070  0.733252    81   
4     81  ...       0.240945    81        0.995452  0.486193  0.782843    81   
5     81  ...       0.243078    81        1.014794  0.474425  0.771826    81   
6     81  ...       0.232490    81        0.973140  0.495097  0.733152    81   
7     81  ...       0.240900    81        0.995319  0.486213  0.782741    81   
8     81  ...       0.243035    81        1.014651  0.474453  0.771711    81   
9     81  ...       0.219616    81        0.984578  0.493940  0.745834    81   
10    81  ...       0.231499    81        1.026308  0.471736  0.817514    81   
11    81  ...       0.242243    81        1.061798  0.451861  0.873710    81   

   explain_time_sec                            
               mean       std    median count  
0          0.000545  0.000555  0.000156    81  
1          0.000541  0.000566  0.000157    81  
2          0.000529  0.000543  0.000159    81  
3          0.002171  0.002296  0.000621    81  
4          0.002191  0.002279  0.000594    81  
5          0.002153  0.002281  0.000595    81  
6          0.002172  0.002306  0.000607    81  
7          0.002170  0.002305  0.000597    81  
8          0.002144  0.002274  0.000594    81  
9          0.000520  0.000540  0.000141    81  
10         0.000518  0.000545  0.000141    81  
11         0.000512  0.000546  0.000138    81  

[12 rows x 46 columns]

,subset,method_a,method_b,contrast_interpretation,metric,metric_role,preferred_direction,n_repeated_pairs,n_condition_clusters,method_a_mean,method_b_mean,a_minus_b_cluster_mean,CI95_low,CI95_high,median_cluster_difference,wilcoxon_p_on_cluster_means
0,all_conditions,HuberRidgeAIME,RidgeAIME,Huber contribution beyond ridge,clean_target_reconstruction_r2,clean-target reconstruction,higher is better,243,81,0.282510,0.264959,1.755093e-02,1.330274e-02,2.226929e-02,1.191958e-02,1.905442e-12
1,all_conditions,HuberRidgeAIME,RidgeAIME,Huber contribution beyond ridge,clean_target_reconstruction_cosine,clean-target reconstruction,higher is better,243,81,0.496700,0.482792,1.390760e-02,1.072236e-02,1.729766e-02,9.584855e-03,7.236609e-12
2,all_conditions,HuberRidgeAIME,RidgeAIME,Huber contribution beyond ridge,operator_cosine_flat,same-method clean-reference operator recovery,higher is better,243,81,0.996602,0.962241,3.436113e-02,2.413986e-02,4.556664e-02,1.576327e-02,4.612818e-12
3,all_conditions,HuberRidgeAIME,RidgeAIME,Huber contribution beyond ridge,operator_topk_jaccard,same-method clean-reference operator recovery,higher is better,243,81,0.824365,0.671526,1.528393e-01,1.195422e-01,1.869706e-01,1.212121e-01,3.077993e-10
4,all_conditions,HuberRidgeAIME,RidgeAIME,Huber contribution beyond ridge,bootstrap_cosine_flat,stability/self-consistency,higher is better,243,81,0.971918,0.948396,2.352203e-02,1.820320e-02,2.927618e-02,1.412522e-02,8.080586e-15
5,all_conditions,HuberRidgeAIME,RidgeAIME,Huber contribution beyond ridge,irrelevant_decoy_mass_ratio,true irrelevant-decoy attribution,lower is better,243,81,0.015412,0.018134,-2.721575e-03,-3.744533e-03,-1.776076e-03,-1.472186e-04,3.455993e-07
6,all_conditions,HuberRidgeAIME,RidgeAIME,Huber contribution beyond ridge,clone_mass_ratio,correlated-clone redistribution,descriptive,243,81,0.153480,0.154119,-6.385849e-04,-1.479306e-03,2.088010e-04,0.000000e+00,1.027561e-01
7,all_conditions,HuberRidgeAIME,RidgeAIME,Huber contribution beyond ridge,log10_cond_reg,conditioning diagnostic,lower is better,243,81,0.288241,0.276405,1.183616e-02,7.786616e-03,1.593286e-02,1.197544e-02,1.843920e-07
8,all_conditions,HuberRidgeAIME,RidgeAIME,Huber contribution beyond ridge,log10_coef_norm,regularization diagnostic,diagnostic only,243,81,0.994370,1.024228,-2.985792e-02,-3.663050e-02,-2.363429e-02,-1.897724e-02,5.778810e-15
9,all_conditions,HuberRidgeAIME,HuberAIME,ridge contribution beyond Huber,clean_target_reconstruction_r2,clean-target reconstruction,higher is better,243,81,0.282510,0.282509,9.213158e-07,-1.174888e-06,2.994229e-06,-2.326106e-08,1.386720e-01


## 4. Main/core figures regenerated from the corrected clean and stress results

In [13]:

# %% [figures]
def save_figure(fig, filename):
    path = MAIN_FIGDIR / filename
    fig.tight_layout()
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.close(fig)
    print("[figure]", path)
    return path

FIGURE_PATHS = []

if len(MAIN_CLEAN_RAW):
    ok = MAIN_CLEAN_RAW[MAIN_CLEAN_RAW["error"].fillna("") == ""]
    methods = AIME_METHODS
    data = [ok.loc[ok["method"] == m, "clean_target_reconstruction_cosine"].dropna().to_numpy() for m in methods]
    fig, ax = plt.subplots(figsize=(7.2, 4.6))
    ax.boxplot(data, labels=methods, showmeans=True)
    ax.set_ylabel("Held-out clean reconstruction cosine")
    ax.set_title("Corrected clean inverse-map benchmark")
    ax.tick_params(axis="x", rotation=20)
    FIGURE_PATHS.append(save_figure(fig, "fig_main_corrected_clean_reconstruction.png"))

if len(MAIN_STRESS_RAW):
    ok = MAIN_STRESS_RAW[MAIN_STRESS_RAW["error"].fillna("") == ""]
    for metric, ylabel, filename, title in [
        ("operator_cosine_flat", "Same-method clean-reference operator cosine", "fig_main_corrected_stress_operator_recovery.png", "Operator recovery under controlled stress"),
        ("irrelevant_decoy_mass_ratio", "True irrelevant-decoy attribution mass", "fig_main_corrected_stress_irrelevant_decoy_mass.png", "Irrelevant-decoy mass under controlled stress"),
    ]:
        agg = ok.groupby(["method", "outlier_rate"], as_index=False)[metric].agg(["mean", "std"]).reset_index()
        fig, ax = plt.subplots(figsize=(7.2, 4.6))
        for method in AIME_METHODS:
            g = agg[agg["method"] == method].sort_values("outlier_rate")
            ax.errorbar(g["outlier_rate"], g["mean"], yerr=g["std"], marker="o", capsize=3, label=method)
        ax.set_xlabel("Injected outlier rate")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.legend(frameon=False, fontsize=8)
        FIGURE_PATHS.append(save_figure(fig, filename))

    cond = ok.groupby("method", as_index=False)[["log10_cond_raw", "log10_cond_reg"]].mean()
    x = np.arange(len(cond))
    fig, ax = plt.subplots(figsize=(7.2, 4.6))
    width = 0.35
    ax.bar(x - width/2, cond["log10_cond_raw"], width, label="raw weighted Gram")
    ax.bar(x + width/2, cond["log10_cond_reg"], width, label="solved system")
    ax.set_xticks(x, cond["method"], rotation=20)
    ax.set_ylabel("Mean log10 condition number")
    ax.set_title("Conditioning diagnostic under corrected inverse mapping")
    ax.legend(frameon=False)
    FIGURE_PATHS.append(save_figure(fig, "fig_main_corrected_conditioning_diagnostic.png"))

print("Generated", len(FIGURE_PATHS), "main/core figures.")


[figure] /Users/takafumi/Documents/Python/HuberRidgeAIME/notebooks/output/main_repro_corrected/figures/fig_main_corrected_clean_reconstruction.png
[figure] /Users/takafumi/Documents/Python/HuberRidgeAIME/notebooks/output/main_repro_corrected/figures/fig_main_corrected_stress_operator_recovery.png
[figure] /Users/takafumi/Documents/Python/HuberRidgeAIME/notebooks/output/main_repro_corrected/figures/fig_main_corrected_stress_irrelevant_decoy_mass.png
[figure] /Users/takafumi/Documents/Python/HuberRidgeAIME/notebooks/output/main_repro_corrected/figures/fig_main_corrected_conditioning_diagnostic.png
Generated 4 main/core figures.


## 5. Australian Credit missing-data sensitivity comparison (corrected inverse map)

In [14]:

# %% [missing-data comparison]
def run_missing_data_comparison():
    out = MAIN_DATADIR / "main_corrected_missing_data_raw.csv"
    cached = load_versioned_csv(out)
    if cached is not None:
        return cached
    if not RUN_MISSING_DATA:
        return pd.DataFrame()
    rows = []
    for protocol in ["impute", "complete_case"]:
        Xdf, ys, raw = process_credit(protocol)
        X = StandardScaler().fit_transform(Xdf.to_numpy(float))
        y = ys.to_numpy(int)
        for rep in range(MISSING_REPEATS):
            for learner in LEARNERS:
                seed = MAIN_SEED + 10000 * rep + 100 * (LEARNERS.index(learner) + 1)
                print(f"[missing] protocol={protocol} {learner} rep={rep}")
                try:
                    model, _, exp_idx, Y_exp, model_info = fit_clean_blackbox(X, y, learner, seed)
                    X_exp, y_exp = X[exp_idx], y[exp_idx]
                    fit_idx, val_idx = split_explanation_rows(y_exp, seed + 17)
                    for method in AIME_METHODS:
                        A, diag, state = fit_inverse_map(
                            X_exp[fit_idx], Y_exp[fit_idx], method,
                            DEFAULT_HUBER_DELTA, DEFAULT_RIDGE_LAMBDA, RESIDUAL_SCALE_MODE,
                        )
                        rows.append({
                            "protocol": protocol, "learner": learner, "repeat": rep,
                            "method": method, "n_after_protocol": len(X),
                            "d_after_encoding": X.shape[1],
                            "raw_missing_cells": int(raw.drop(columns=["target"]).isna().sum().sum()),
                            **model_info, **diag,
                            **inverse_reconstruction_metrics(state, Y_exp[val_idx], X_exp[val_idx], "clean_target_"),
                            **bootstrap_inverse_stability(
                                X_exp[fit_idx], Y_exp[fit_idx], method,
                                DEFAULT_HUBER_DELTA, DEFAULT_RIDGE_LAMBDA,
                                A0=A, B=MISSING_BOOTSTRAPS, seed=seed + 29,
                            ),
                            **irrelevant_decoy_metrics(
                                X_exp[fit_idx], Y_exp[fit_idx], method,
                                DEFAULT_HUBER_DELTA, DEFAULT_RIDGE_LAMBDA,
                                MISSING_DECOY_TRIALS, seed + 61,
                            ),
                            "error": "",
                        })
                except Exception as e:
                    rows.append({"protocol": protocol, "learner": learner, "repeat": rep, "method": "FAILED", "error": str(e)})
    df = pd.DataFrame(rows)
    if len(df):
        df["log10_cond_reg"] = np.log10(df["cond_reg"].clip(lower=1e-300))
        df["log10_coef_norm"] = np.log10(df["coef_fro_norm"].clip(lower=1e-300))
        save_csv(df, out)
    return df

def summarize_missing_data(df):
    ok = df[df["error"].fillna("") == ""].copy()
    metrics = [
        "n_after_protocol", "d_after_encoding", "model_test_accuracy",
        "clean_target_reconstruction_r2", "clean_target_reconstruction_cosine",
        "bootstrap_cosine_flat", "irrelevant_decoy_mass_ratio",
        "log10_cond_reg", "log10_coef_norm",
    ]
    summary = ok.groupby(["protocol", "learner", "method"])[metrics].agg(["mean", "std", "median", "count"]).reset_index()
    save_table_bundle(
        summary, "table_main_corrected_missing_data_comparison",
        "Australian Credit complete-case versus imputation sensitivity analysis under the corrected inverse orientation. Coefficient norm is diagnostic only.",
        "tab:main_corrected_missing_data",
    )
    return summary

MAIN_MISSING_RAW = run_missing_data_comparison()
MAIN_MISSING_SUMMARY = summarize_missing_data(MAIN_MISSING_RAW) if len(MAIN_MISSING_RAW) else pd.DataFrame()
display(MAIN_MISSING_SUMMARY)


[missing] protocol=impute lgbm rep=0
[missing] protocol=impute svm rep=0
[missing] protocol=impute mlp rep=0
[missing] protocol=impute lgbm rep=1
[missing] protocol=impute svm rep=1
[missing] protocol=impute mlp rep=1
[missing] protocol=impute lgbm rep=2
[missing] protocol=impute svm rep=2
[missing] protocol=impute mlp rep=2
[missing] protocol=complete_case lgbm rep=0
[missing] protocol=complete_case svm rep=0
[missing] protocol=complete_case mlp rep=0
[missing] protocol=complete_case lgbm rep=1
[missing] protocol=complete_case svm rep=1
[missing] protocol=complete_case mlp rep=1
[missing] protocol=complete_case lgbm rep=2
[missing] protocol=complete_case svm rep=2
[missing] protocol=complete_case mlp rep=2
[csv] /Users/takafumi/Documents/Python/HuberRidgeAIME/notebooks/output/main_repro_corrected/data/main_corrected_missing_data_raw.csv
[csv] /Users/takafumi/Documents/Python/HuberRidgeAIME/notebooks/output/main_repro_corrected/data/table_main_corrected_missing_data_comparison.csv
[tex

protocol learner          method n_after_protocol                    \
                                                      mean  std median count   
0   complete_case    lgbm            AIME            653.0  0.0  653.0     3   
1   complete_case    lgbm       HuberAIME            653.0  0.0  653.0     3   
2   complete_case    lgbm  HuberRidgeAIME            653.0  0.0  653.0     3   
3   complete_case    lgbm       RidgeAIME            653.0  0.0  653.0     3   
4   complete_case     mlp            AIME            653.0  0.0  653.0     3   
5   complete_case     mlp       HuberAIME            653.0  0.0  653.0     3   
6   complete_case     mlp  HuberRidgeAIME            653.0  0.0  653.0     3   
7   complete_case     mlp       RidgeAIME            653.0  0.0  653.0     3   
8   complete_case     svm            AIME            653.0  0.0  653.0     3   
9   complete_case     svm       HuberAIME            653.0  0.0  653.0     3   
10  complete_case     svm  HuberRidgeAIME            653.0  0.0  653.0     3   
11  complete_case     svm       RidgeAIME            653.0  0.0  653.0     3   
12         impute    lgbm            AIME            690.0  0.0  690.0     3   
13         impute    lgbm       HuberAIME            690.0  0.0  690.0     3   
14         impute    lgbm  HuberRidgeAIME            690.0  0.0  690.0     3   
15         impute    lgbm       RidgeAIME            690.0  0.0  690.0     3   
16         impute     mlp            AIME            690.0  0.0  690.0     3   
17         impute     mlp       HuberAIME            690.0  0.0  690.0     3   
18         impute     mlp  HuberRidgeAIME            690.0  0.0  690.0     3   
19         impute     mlp       RidgeAIME            690.0  0.0  690.0     3   
20         impute     svm            AIME            690.0  0.0  690.0     3   
21         impute     svm       HuberAIME            690.0  0.0  690.0     3   
22         impute     svm  HuberRidgeAIME            690.0  0.0  690.0     3   
23         impute     svm       RidgeAIME            690.0  0.0  690.0     3   

   d_after_encoding              ... irrelevant_decoy_mass_ratio        \
               mean  std median  ...                      median count   
0              37.0  0.0   37.0  ...                    0.035296     3   
1              37.0  0.0   37.0  ...                    0.036012     3   
2              37.0  0.0   37.0  ...                    0.036012     3   
3              37.0  0.0   37.0  ...                    0.035296     3   
4              37.0  0.0   37.0  ...                    0.028425     3   
5              37.0  0.0   37.0  ...                    0.028299     3   
6              37.0  0.0   37.0  ...                    0.028300     3   
7              37.0  0.0   37.0  ...                    0.028426     3   
8              37.0  0.0   37.0  ...                    0.037103     3   
9              37.0  0.0   37.0  ...                    0.035244     3   
10             37.0  0.0   37.0  ...                    0.035245     3   
11             37.0  0.0   37.0  ...                    0.037105     3   
12             42.0  0.0   42.0  ...                    0.030136     3   
13             42.0  0.0   42.0  ...                    0.033010     3   
14             42.0  0.0   42.0  ...                    0.033011     3   
15             42.0  0.0   42.0  ...                    0.030136     3   
16             42.0  0.0   42.0  ...                    0.020920     3   
17             42.0  0.0   42.0  ...                    0.018034     3   
18             42.0  0.0   42.0  ...                    0.018035     3   
19             42.0  0.0   42.0  ...                    0.020921     3   
20             42.0  0.0   42.0  ...                    0.038961     3   
21             42.0  0.0   42.0  ...                    0.034571     3   
22             42.0  0.0   42.0  ...                    0.034572     3   
23             42.0  0.0   42.0  ...                    0.038962     3   

   

## 6. Equation-consistent runtime scaling

In [15]:

# %% [runtime scaling]
def generate_inverse_runtime_data(n=1000, d=100, C=3, outlier_rate=0.05, seed=42):
    rng = np.random.default_rng(seed)
    Y = softmax(rng.normal(size=(n, C)), axis=1)
    B_true = rng.normal(size=(C, d)) / np.sqrt(max(1, C))
    X_clean = Y @ B_true + 0.05 * rng.normal(size=(n, d))
    X_obs, _, _ = inject_outliers_with_mask(X_clean, rate=outlier_rate, scale=6.0, seed=seed + 1)
    return X_obs, Y

def run_runtime_scaling():
    out = MAIN_DATADIR / "main_corrected_runtime_scaling_raw.csv"
    cached = load_versioned_csv(out)
    if cached is not None:
        return cached
    if not RUN_RUNTIME:
        return pd.DataFrame()
    rows = []
    for n in RUNTIME_N_VALUES:
        for d in RUNTIME_D_VALUES:
            for rep in range(RUNTIME_REPEATS):
                X, Y = generate_inverse_runtime_data(n=n, d=d, C=3, seed=MAIN_SEED + n + d + 1000 * rep)
                for method in AIME_METHODS:
                    t0 = time.perf_counter()
                    _, diag, _ = fit_inverse_map(
                        X, Y, method, DEFAULT_HUBER_DELTA, DEFAULT_RIDGE_LAMBDA, RESIDUAL_SCALE_MODE
                    )
                    elapsed = time.perf_counter() - t0
                    rows.append({
                        "n": n, "d": d, "C": Y.shape[1], "repeat": rep,
                        "method": method, "runtime_sec": elapsed, "n_times_d": n * d,
                        **diag, "error": "",
                    })
                    print(f"[runtime] n={n} d={d} {method}: {elapsed:.4f}s")
    df = pd.DataFrame(rows)
    save_csv(df, out)
    return df

def summarize_runtime(df):
    ok = df[df["error"].fillna("") == ""].copy()
    summary = ok.groupby("method", as_index=False).agg(
        mean_runtime_sec=("runtime_sec", "mean"),
        median_runtime_sec=("runtime_sec", "median"),
        max_runtime_sec=("runtime_sec", "max"),
        mean_iterations=("mean_iter", "mean"),
    )
    save_table_bundle(
        summary, "table_main_corrected_runtime_scaling",
        "Runtime scaling of equation-consistent AIME-family inverse-map solvers on controlled synthetic matrices. Huber variants require IRLS iterations; the ridge term adds negligible overhead relative to the corresponding Huber solve.",
        "tab:main_corrected_runtime",
    )
    plot_df = ok.groupby(["method", "n_times_d"], as_index=False)["runtime_sec"].mean()
    fig, ax = plt.subplots(figsize=(7.2, 4.6))
    for method in AIME_METHODS:
        g = plot_df[plot_df["method"] == method].sort_values("n_times_d")
        ax.plot(g["n_times_d"], g["runtime_sec"], marker="o", label=method)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("n × d (log scale)"); ax.set_ylabel("Runtime seconds (log scale)")
    ax.set_title("Corrected AIME-family runtime scaling")
    ax.legend(frameon=False, fontsize=8)
    save_figure(fig, "fig_main_corrected_runtime_scaling.png")
    return summary

MAIN_RUNTIME_RAW = run_runtime_scaling()
MAIN_RUNTIME_SUMMARY = summarize_runtime(MAIN_RUNTIME_RAW) if len(MAIN_RUNTIME_RAW) else pd.DataFrame()
display(MAIN_RUNTIME_SUMMARY)


[runtime] n=500 d=20 AIME: 0.0004s
[runtime] n=500 d=20 HuberAIME: 0.0009s
[runtime] n=500 d=20 RidgeAIME: 0.0002s
[runtime] n=500 d=20 HuberRidgeAIME: 0.0005s
[runtime] n=500 d=20 AIME: 0.0002s
[runtime] n=500 d=20 HuberAIME: 0.0005s
[runtime] n=500 d=20 RidgeAIME: 0.0002s
[runtime] n=500 d=20 HuberRidgeAIME: 0.0005s
[runtime] n=500 d=20 AIME: 0.0002s
[runtime] n=500 d=20 HuberAIME: 0.0005s
[runtime] n=500 d=20 RidgeAIME: 0.0002s
[runtime] n=500 d=20 HuberRidgeAIME: 0.0005s
[runtime] n=500 d=100 AIME: 0.0003s
[runtime] n=500 d=100 HuberAIME: 0.0008s
[runtime] n=500 d=100 RidgeAIME: 0.0003s
[runtime] n=500 d=100 HuberRidgeAIME: 0.0008s
[runtime] n=500 d=100 AIME: 0.0003s
[runtime] n=500 d=100 HuberAIME: 0.0008s
[runtime] n=500 d=100 RidgeAIME: 0.0004s
[runtime] n=500 d=100 HuberRidgeAIME: 0.0009s
[runtime] n=500 d=100 AIME: 0.0003s
[runtime] n=500 d=100 HuberAIME: 0.0008s
[runtime] n=500 d=100 RidgeAIME: 0.0003s
[runtime] n=500 d=100 HuberRidgeAIME: 0.0008s
[runtime] n=500 d=300 AIME: 

,method,mean_runtime_sec,median_runtime_sec,max_runtime_sec,mean_iterations
0,AIME,0.000969,0.000547,0.004051,1.000000
1,HuberAIME,0.002629,0.001625,0.010731,4.944444
2,HuberRidgeAIME,0.002532,0.001530,0.010220,4.944444
3,RidgeAIME,0.000924,0.000507,0.003690,1.000000


## 7. Provenance, validation gates, and final package

In [16]:

# %% [provenance, validation, and package]
def write_main_readme():
    text = f"""# HuberRidgeAIME corrected main reproducibility outputs

Pipeline version: `{PIPELINE_VERSION}`

## Scientific correction
All AIME-family estimators in this release solve `X ≈ Y A^T`.  Huber residuals are row-RMS input-space residuals.  Coefficient norm is reported only as a regularization diagnostic.

## Division of responsibility between notebooks
- `HuberRidgeAIME_Main_Reproducibility_CORRECTED.ipynb`: dataset audit, corrected clean benchmark, all-method controlled stress, missing-data sensitivity, runtime, and core artifacts.
- `HuberRidgeAIME_Reviewer5_FullyCorrected_Experiments_FINAL.ipynb`: known-ground-truth faithfulness, focused HRA-versus-RidgeAIME experiments, lambda/delta sensitivity, LIME/TreeSHAP stress comparison, and spectral diagnostics.

The previous main-analysis implementation must not be used for current-manuscript numerical results.
"""
    path = MAIN_OUTDIR / "README_main_corrected.md"
    path.write_text(text, encoding="utf-8")
    return path

def write_main_requirements():
    path = MAIN_OUTDIR / "requirements_main_corrected.txt"
    path.write_text("""numpy
pandas
scipy
scikit-learn
matplotlib
lightgbm
nbformat
""", encoding="utf-8")
    return path

def write_main_run_configuration():
    config = {
        "pipeline_version": PIPELINE_VERSION,
        "seed": MAIN_SEED,
        "quick_test": QUICK_TEST,
        "force_recompute": FORCE_RECOMPUTE,
        "datasets": DATASETS,
        "learners": LEARNERS,
        "max_n_per_dataset": MAX_N_PER_DATASET,
        "default_ridge_lambda": DEFAULT_RIDGE_LAMBDA,
        "default_huber_delta": DEFAULT_HUBER_DELTA,
        "residual_scale_mode": RESIDUAL_SCALE_MODE,
        "clean_repeats": CLEAN_REPEATS,
        "stress_repeats": STRESS_REPEATS,
        "stress_outlier_levels": STRESS_OUTLIER_LEVELS,
        "stress_clone_fractions": STRESS_CLONE_FRACS,
        "missing_repeats": MISSING_REPEATS,
        "runtime_n_values": RUNTIME_N_VALUES,
        "runtime_d_values": RUNTIME_D_VALUES,
        "runtime_repeats": RUNTIME_REPEATS,
    }
    path = MAIN_OUTDIR / "main_corrected_run_configuration.json"
    path.write_text(json.dumps(config, indent=2, default=str), encoding="utf-8")
    return path

def write_main_environment_manifest():
    rows = []
    for mod in ["numpy", "pandas", "scipy", "sklearn", "matplotlib", "lightgbm"]:
        try:
            m = __import__(mod); version = getattr(m, "__version__", "available")
        except Exception as e:
            version = f"not available: {e}"
        rows.append({"package": mod, "version": version})
    rows += [
        {"package": "python", "version": sys.version.replace("\n", " ")},
        {"package": "platform", "version": platform.platform()},
        {"package": "pipeline_version", "version": PIPELINE_VERSION},
        {"package": "inverse_orientation", "version": "X ~= Y A^T"},
    ]
    df = pd.DataFrame(rows)
    save_csv(df, MAIN_DATADIR / "main_corrected_environment_manifest.csv")
    return df

def build_main_provenance():
    rows = [
        {"artifact": "table_main_corrected_dataset_audit.tex", "source": "public dataset loaders", "purpose": "dataset validity and minor-point diagnostics"},
        {"artifact": "table_main_corrected_clean_benchmark_summary.tex", "source": "main_corrected_clean_benchmark_raw.csv", "purpose": "corrected clean AIME-family benchmark"},
        {"artifact": "table_main_corrected_stress_summary.tex", "source": "main_corrected_full_stress_raw.csv", "purpose": "corrected all-method controlled stress results"},
        {"artifact": "table_main_corrected_pairwise_cluster_ci.tex", "source": "main_corrected_full_stress_raw.csv", "purpose": "condition-cluster bootstrap method contrasts"},
        {"artifact": "table_main_corrected_missing_data_comparison.tex", "source": "main_corrected_missing_data_raw.csv", "purpose": "complete-case versus imputation sensitivity"},
        {"artifact": "table_main_corrected_runtime_scaling.tex", "source": "main_corrected_runtime_scaling_raw.csv", "purpose": "equation-consistent runtime scaling"},
        {"artifact": "fig_main_corrected_*.png", "source": "corrected clean/stress/runtime CSVs", "purpose": "main/core figures"},
    ]
    df = pd.DataFrame(rows)
    save_csv(df, MAIN_DATADIR / "main_corrected_figure_table_provenance.csv")
    return df

def finite_columns(df, cols):
    if df.empty: return False
    return bool(np.isfinite(df[cols].to_numpy(float)).all())

def validate_main_outputs():
    checks = {}
    checks["unit_tests_pass"] = bool(MAIN_UNIT_TESTS[["aime_orientation_pass", "large_delta_equivalence_pass", "outlier_downweight_pass"]].all(axis=None))
    expected_audit_datasets = set(DATASETS) if QUICK_TEST else {"breast_cancer", "credit_approval", "har"}
    checks["dataset_audit_has_expected_datasets"] = set(MAIN_DATASET_AUDIT["dataset"]) == expected_audit_datasets

    expected_clean = len(DATASETS) * len(LEARNERS) * CLEAN_REPEATS * len(AIME_METHODS) if RUN_CLEAN_BENCHMARK else 0
    expected_stress = len(DATASETS) * len(LEARNERS) * STRESS_REPEATS * len(STRESS_OUTLIER_LEVELS) * len(STRESS_CLONE_FRACS) * len(AIME_METHODS) if RUN_FULL_STRESS else 0
    expected_missing = 2 * len(LEARNERS) * MISSING_REPEATS * len(AIME_METHODS) if RUN_MISSING_DATA else 0
    expected_runtime = len(RUNTIME_N_VALUES) * len(RUNTIME_D_VALUES) * RUNTIME_REPEATS * len(AIME_METHODS) if RUN_RUNTIME else 0

    checks["clean_expected_rows"] = (not RUN_CLEAN_BENCHMARK) or len(MAIN_CLEAN_RAW) == expected_clean
    checks["stress_expected_rows"] = (not RUN_FULL_STRESS) or len(MAIN_STRESS_RAW) == expected_stress
    checks["runtime_expected_rows"] = (not RUN_RUNTIME) or len(MAIN_RUNTIME_RAW) == expected_runtime
    if RUN_MISSING_DATA:
        checks["missing_expected_rows"] = len(MAIN_MISSING_RAW) == 2 * len(LEARNERS) * MISSING_REPEATS * len(AIME_METHODS)
    else:
        checks["missing_expected_rows"] = True

    for name, df in [("clean", MAIN_CLEAN_RAW), ("stress", MAIN_STRESS_RAW), ("missing", MAIN_MISSING_RAW), ("runtime", MAIN_RUNTIME_RAW)]:
        if len(df):
            checks[f"{name}_no_errors"] = bool((df["error"].fillna("") == "").all())
            if "orientation" in df.columns:
                checks[f"{name}_inverse_orientation"] = bool((df["orientation"] == "X ~= Y A^T").all())
        else:
            checks[f"{name}_no_errors"] = True
            checks[f"{name}_inverse_orientation"] = True

    if len(MAIN_STRESS_RAW):
        checks["stress_has_all_methods"] = set(AIME_METHODS).issubset(set(MAIN_STRESS_RAW["method"]))
        checks["pairwise_ci_nonempty"] = len(MAIN_PAIRWISE_CI) > 0
        checks["stress_key_metrics_finite"] = finite_columns(MAIN_STRESS_RAW, [
            "clean_target_reconstruction_cosine", "operator_cosine_flat",
            "irrelevant_decoy_mass_ratio", "log10_cond_reg", "log10_coef_norm",
        ])
    else:
        checks["stress_has_all_methods"] = True
        checks["pairwise_ci_nonempty"] = True
        checks["stress_key_metrics_finite"] = True

    failed = [k for k, v in checks.items() if not bool(v)]
    report = {
        "pipeline_version": PIPELINE_VERSION,
        "quick_test": QUICK_TEST,
        "expected_rows": {"clean": expected_clean, "stress": expected_stress, "missing": expected_missing, "runtime": expected_runtime},
        "observed_rows": {"clean": len(MAIN_CLEAN_RAW), "stress": len(MAIN_STRESS_RAW), "missing": len(MAIN_MISSING_RAW), "runtime": len(MAIN_RUNTIME_RAW)},
        "checks": checks,
        "failed": failed,
    }
    path = MAIN_OUTDIR / "main_corrected_validation_report.json"
    path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    if failed:
        raise AssertionError(f"Main corrected validation failed: {failed}")
    print("All main-corrected validation gates passed.")
    return report

def build_manifest_and_zip():
    records = []
    for p in sorted(MAIN_OUTDIR.rglob("*")):
        if p.is_file() and not p.name.endswith(".zip") and p.name != "main_corrected_artifact_manifest.csv":
            records.append({"relative_path": str(p.relative_to(MAIN_OUTDIR)), "bytes": p.stat().st_size, "sha256": file_sha256(p)})
    manifest = pd.DataFrame(records)
    save_csv(manifest, MAIN_OUTDIR / "main_corrected_artifact_manifest.csv")
    zip_path = MAIN_OUTDIR / "HuberRidgeAIME_Main_Reproducibility_CORRECTED_outputs.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in MAIN_OUTDIR.rglob("*"):
            if p.is_file() and p.resolve() != zip_path.resolve():
                zf.write(p, p.relative_to(MAIN_OUTDIR))
    print("[zip]", zip_path)
    return zip_path

README_PATH = write_main_readme()
REQ_PATH = write_main_requirements()
CONFIG_PATH = write_main_run_configuration()
MAIN_ENVIRONMENT = write_main_environment_manifest()
MAIN_PROVENANCE = build_main_provenance()
MAIN_VALIDATION = validate_main_outputs()
MAIN_ZIP_PATH = build_manifest_and_zip()

print("\nOutput inventory")
for folder in [MAIN_DATADIR, MAIN_TABLEDIR, MAIN_FIGDIR, MAIN_LOGDIR]:
    files = sorted(p.name for p in folder.glob("*") if p.is_file())
    print(f"{folder.name}/: {len(files)} files")
    for name in files:
        print(" -", name)
print("ZIP:", MAIN_ZIP_PATH)

[csv] /Users/takafumi/Documents/Python/HuberRidgeAIME/notebooks/output/main_repro_corrected/data/main_corrected_environment_manifest.csv
[csv] /Users/takafumi/Documents/Python/HuberRidgeAIME/notebooks/output/main_repro_corrected/data/main_corrected_figure_table_provenance.csv
All main-corrected validation gates passed.
[csv] /Users/takafumi/Documents/Python/HuberRidgeAIME/notebooks/output/main_repro_corrected/main_corrected_artifact_manifest.csv
[zip] /Users/takafumi/Documents/Python/HuberRidgeAIME/notebooks/output/main_repro_corrected/HuberRidgeAIME_Main_Reproducibility_CORRECTED_outputs.zip

Output inventory
data/: 13 files
 - main_corrected_clean_benchmark_raw.csv
 - main_corrected_environment_manifest.csv
 - main_corrected_figure_table_provenance.csv
 - main_corrected_full_stress_raw.csv
 - main_corrected_inverse_map_unit_tests.csv
 - main_corrected_missing_data_raw.csv
 - main_corrected_runtime_scaling_raw.csv
 - table_main_corrected_clean_benchmark_summary.csv
 - table_main_corre